# Whisper fine-tune for Hindi-English (Hinglish) speech

**Kaggle settings (right sidebar):** Accelerator → **GPU T4 x2** (or P100) · Internet → **On** · Persistence → Files (optional).
Optional: add a Kaggle Secret named `HF_TOKEN` (Add-ons → Secrets) to push models to Hugging Face or use gated datasets.

In [ ]:
%env CUDA_VISIBLE_DEVICES=0
# one GPU is enough; avoids multi-GPU DataParallel surprises
!nvidia-smi

In [ ]:
!git clone -q https://github.com/Sidvortex/whisper-hinglish.git || (cd whisper-hinglish && git pull -q)
%cd whisper-hinglish
!pip install -q -U -r requirements.txt

In [ ]:
# optional: log in to Hugging Face with a Kaggle secret
try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    login(UserSecretsClient().get_secret('HF_TOKEN'))
except Exception as e:
    print('No HF_TOKEN secret, continuing anonymously:', type(e).__name__)

In [ ]:
!apt-get -qq install -y libsndfile1 > /dev/null 2>&1 || true

## 1. Smoke run (≈10 min)

In [ ]:
!python train.py --set train.max_steps=60 train.eval_steps=30 train.save_steps=30 data.max_eval_samples=32 train.output_dir=outputs/smoke

## 2. Full fine-tune of whisper-small (≈3–4 h). If Kaggle's 12 h limit is a worry, lower `train.max_steps`.

In [ ]:
!python train.py --config configs/whisper_small.yaml

## 3. Base vs fine-tuned on the test split (overall + per language)

In [ ]:
!python eval_model.py --config configs/whisper_small.yaml --limit 800
import json; print(json.dumps(json.load(open('outputs/whisper-small-hinglish/test_results.json')), indent=2, ensure_ascii=False))

In [ ]:
from IPython.display import Markdown
Markdown(open('outputs/whisper-small-hinglish/test_samples.md').read())

## 4. Save the model

In [ ]:
!cd outputs/whisper-small-hinglish && zip -qr /kaggle/working/whisper_small_hinglish.zip final test_results.json test_samples.md
!ls -lh /kaggle/working/*.zip

Optional: push to the Hub (needs the HF_TOKEN secret)

In [ ]:
# from transformers import WhisperForConditionalGeneration, WhisperProcessor
# m = WhisperForConditionalGeneration.from_pretrained('outputs/whisper-small-hinglish/final')
# p = WhisperProcessor.from_pretrained('outputs/whisper-small-hinglish/final')
# m.push_to_hub('Sidvortex/whisper-small-hinglish'); p.push_to_hub('Sidvortex/whisper-small-hinglish')